# RNN for captions

In [90]:
import numpy as np
import pandas as pd

### Load split data

In [91]:
train_df = pd.read_csv(
    "data/train_df.csv",
    parse_dates=["publish_timestamp"]  # converts datestime string into datetime object
)
test_df = pd.read_csv(
    "data/test_df.csv",
    parse_dates=["publish_timestamp"]
)

In [92]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1471 entries, 0 to 1470
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   1471 non-null   int64         
 1   following                 1471 non-null   int64         
 2   publish_timestamp         1471 non-null   datetime64[ns]
 3   has_location              1471 non-null   int64         
 4   is_carousel               1471 non-null   int64         
 5   num_images                1471 non-null   int64         
 6   is_sponsored              1471 non-null   int64         
 7   image_path                1471 non-null   object        
 8   caption                   1443 non-null   object        
 9   follower_following_ratio  1471 non-null   float64       
 10  hour                      1471 non-null   int64         
 11  day                       1471 non-null   object        
 12  is_weekend          

# LSTM

### Fill any missing captions

In [93]:
train_df["caption"] = train_df["caption"].fillna("")
test_df["caption"]  = test_df["caption"].fillna("")


### Tokenize captions & build vocabulary, using train_df

In [94]:
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import re

# Simple tokenizer: split on spaces, remove non-alphanumeric chars
def tokenize(text):
    if not isinstance(text, str):
        return []
    # lowercase and keep only words (a-z, 0-9)
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

# Build vocab
all_tokens = []
for caption in train_df['caption']:
    all_tokens.extend(tokenize(caption))

# Count frequencies and build vocab
counter = Counter(all_tokens)
vocab = {"<PAD>": 0, "<UNK>": 1}  # reserve 0 for padding, 1 for unknown
for i, word in enumerate(counter.keys(), start=2):
    vocab[word] = i

vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")


Vocab size: 9991


### Encode train_df captions to sequences and add padding

In [ ]:
max_len = 64  # choose based on caption length distribution

def encode_caption(caption, vocab, max_len=max_len):
    tokens = tokenize(caption)
    seq = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    # pad or truncate
    if len(seq) < max_len:
        seq += [vocab["<PAD>"]] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq


### Apply encoding to both train_df and test_df

In [96]:
train_df["caption_seq"] = train_df["caption"].apply(
    lambda x: encode_caption(x, vocab, max_len)
)

test_df["caption_seq"] = test_df["caption"].apply(
    lambda x: encode_caption(x, vocab, max_len)
)

In [97]:
X_train = train_df["caption_seq"].tolist()
y_train = train_df["engagement_label"].values

X_test = test_df["caption_seq"].tolist()
y_test = test_df["engagement_label"].values

### Caption dataset

In [98]:
class CaptionDataset(Dataset):
    def __init__(self, sequences, labels):
        self.X = torch.tensor(sequences, dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = CaptionDataset(X_train, y_train)
test_ds  = CaptionDataset(X_test, y_test)


#### DataLoaders

In [99]:
batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)


### Caption RNN model using PyTorch LSTM model

In [100]:
import torch.nn as nn

class CaptionRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.5):
        super().__init__()

        # Embedding layer for token-based text classification
        # Maps token id to dense vector
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Single-layer LSTM
        # each timestep is an embedding vector, which has size of embed_dim
        # returns: output, (hidden, cell)
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        # Dropout after LSTM
        self.dropout = nn.Dropout(dropout)
        # Optional hidden layer
        self.fc_hidden = nn.Linear(hidden_dim, hidden_dim // 2)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim // 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)                     # (batch_size, seq_len, embed_dim)
        _, (hidden, _) = self.lstm(emb)             # lstm output: output, (hidden, cell). output: (batch, seq_len, hidden_dim). hidden: (num_layers*num_directions, batch_size, hidden_dim).
        h = hidden.squeeze(0)                       # (batch_size, hidden_dim), can squeeze because num_layers*num_directions = 1
        h = self.dropout(h)                         # apply dropout
        h = self.fc_hidden(h)                       # optional hidden layer
        h = self.relu(h)                            # non-linearity
        h = self.dropout(h)                         # another dropout
        out = self.fc_out(h)                        # (batch_size, num_classes), output logits
        return out

# Hyperparameters
embed_dim = 50
hidden_dim = 64
num_classes = 3
dropout = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CaptionRNN(vocab_size, embed_dim, hidden_dim, num_classes, dropout).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


### Train and test loop

In [101]:

# training loop
def train_epoch(loader):
    # Put model in training mode
    model.train()

    # Initialize accumulator for total loss
    total_loss = 0

    # Loops over batches in training set
    for batch in loader:
        # Detect if batch is a dict (BERT-style) or tuple (LSTM/MLP/CNN)
        if isinstance(batch, dict):
            # BERT-style
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            # total_loss += loss.item()
            # correct += (logits.argmax(dim=1) == labels).sum().item()
            # total += labels.size(0)

        else:
            # Tuple-style: (X, y) for LSTM, MLP, CNN
            X, y = batch

            # Move x y to same device
            X, y = X.to(device), y.to(device)

            # Clear old gradients from previous batch
            optimizer.zero_grad()

            # Feeds batch into model, perform forward pass, output logits
            logits = model(X)
            # logits = model(input_ids=X["input_ids"], attention_mask=X["attention_mask"])


            # Compute loss for this batch
            loss = criterion(logits, y)

            # Performs backpropagation: computes gradients of the loss with respect to each model parameter.
            loss.backward()

            # Updates model parameters using those gradients
            optimizer.step()

        # Adds the numeric loss value (converted from a tensor with .item()) to the total loss accumulator.
        total_loss += loss.item()

    # Average loss = total loss / number of batches
    avg_loss = total_loss / len(loader)
    
    return avg_loss




In [102]:
def test_epoch(loader):
    # Set model to evaluation mode. Disables dropout, freezes batchnorm stats
    model.eval()

    # Initialize accumulator for total loss
    total_loss = 0

    with torch.no_grad(): # Disables gradient tracking inside this block.
    # This saves memory and time, since we don’t need gradients for validation.
    # It also ensures the model’s weights won’t accidentally be updated.
        for batch in loader:
            if isinstance(batch, dict):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                y = labels
                # total_loss += criterion(logits, labels).item()
            else:
                X, y = batch
                X, y = X.to(device), y.to(device)
    
                logits = model(X)
                # logits = model(input_ids=X["input_ids"], attention_mask=X["attention_mask"])


            total_loss += criterion(logits, y).item()

    # Compute average loss
    avg_loss = total_loss / len(loader)
    
    return avg_loss

In [103]:
from sklearn.metrics import confusion_matrix, f1_score

def evaluate_metrics(loader, model):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, dict):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                y = labels
            else:
                X, y = batch
                X, y = X.to(device), y.to(device)

                logits = model(X)
                # logits = model(input_ids=X["input_ids"], attention_mask=X["attention_mask"])

            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Accuracy
    correct = sum(p == y for p, y in zip(all_preds, all_labels))
    accuracy = correct / len(all_labels)

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Macro F1
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return accuracy, cm, macro_f1


In [104]:
for epoch in range(10):
    train_loss = train_epoch(train_loader)
    test_loss  = test_epoch(test_loader)

    # Metrics
    train_acc, train_cm, train_macro_f1 = evaluate_metrics(train_loader, model)
    test_acc, test_cm, test_macro_f1 = evaluate_metrics(test_loader, model)

    print(f"\nEpoch {epoch+1}")
    print(f"Train   | loss={train_loss:.4f}, acc={train_acc:.3f}, macro-F1={train_macro_f1:.3f}")
    print(f"Test    | loss={test_loss:.4f}, acc={test_acc:.3f}, macro-F1={test_macro_f1:.3f}")

    print("Train Confusion Matrix:")
    print(train_cm)

    print("Test Confusion Matrix:")
    print(test_cm)



Epoch 1
Train   | loss=1.0963, acc=0.409, macro-F1=0.322
Test    | loss=1.0984, acc=0.370, macro-F1=0.297
Train Confusion Matrix:
[[ 70  56 336]
 [ 28  55 413]
 [ 18  18 477]]
Test Confusion Matrix:
[[ 14  19 114]
 [  5  17  92]
 [  2   5 108]]

Epoch 2
Train   | loss=1.0915, acc=0.438, macro-F1=0.360
Test    | loss=1.0966, acc=0.380, macro-F1=0.305
Train Confusion Matrix:
[[130  26 306]
 [ 56  39 401]
 [ 31   7 475]]
Test Confusion Matrix:
[[ 26  15 106]
 [ 11   9  94]
 [  4   3 108]]

Epoch 3
Train   | loss=1.0736, acc=0.451, macro-F1=0.370
Test    | loss=1.0732, acc=0.410, macro-F1=0.340
Train Confusion Matrix:
[[221   9 232]
 [145  14 337]
 [ 76   9 428]]
Test Confusion Matrix:
[[55 10 82]
 [23  4 87]
 [15  5 95]]

Epoch 4
Train   | loss=1.0556, acc=0.460, macro-F1=0.387
Test    | loss=1.0665, acc=0.410, macro-F1=0.339
Train Confusion Matrix:
[[193  15 254]
 [108  32 356]
 [ 53   9 451]]
Test Confusion Matrix:
[[ 49   8  90]
 [ 20   5  89]
 [ 10   5 100]]

Epoch 5
Train   | loss=1

# BERT

### Import and initialize tokenizer

In [105]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

#### Sample tokenization

In [106]:
# Example for a single caption
caption = "This is an example post caption!"
encoded = tokenizer(
    caption,
    padding='max_length',   # pad all sequences to max_length
    truncation=True,        # truncate if longer than max_length
    max_length=20,          # choose based on caption lengths 
    return_tensors='pt'     # return PyTorch tensors
)

print(encoded)


{'input_ids': tensor([[ 101, 1188, 1110, 1126, 1859, 2112, 6707, 2116,  106,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}


input_ids:
* Each number is the ID of a token in BERT’s vocabulary.
* 101 → [CLS] token, added automatically at the start for classification tasks.
* 102 → [SEP] token, added automatically at the end.
* 0 → [PAD] token, added because your caption is shorter than max_length=20.
* Numbers like 1188, 1110, ... → the IDs of actual words/subwords from the caption.
* Shape: (1, 20) → batch size 1, sequence length 20.

token_type_ids:
* 0: all tokens belong to sentence A
* for single-sentence classification tasks, all values are 0.

attention_mask:
* 1: real tokens, 0: padding

### Tokenize train and test sets separately

Since BERT's vocab is fixed, we encode each set instead of fitting the tokenizer on train only.

In [107]:
train_encodings = tokenizer(
    train_df['caption'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=64,          # choose based on caption lengths (median 21, but max 400 words)
    return_tensors='pt'
)

test_encodings = tokenizer(
    test_df['caption'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=64,          # choose based on caption lengths (median 21, but max 400 words)
    return_tensors='pt'
)


In [108]:
y_train = train_df["engagement_label"].values

y_test = test_df["engagement_label"].values

### PyTorch dataset

In [109]:
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings  # a dict from tokenizer
        self.labels = labels

    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        # Return a dict with each item converted to tensor
        item = {key: val[idx].detach().clone() for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(train_encodings, y_train)
test_ds  = TextDataset(test_encodings, y_test)


### DataLoader

In [110]:

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [111]:
from transformers import BertModel
import torch.nn as nn

# Load pretrained model
bert_model = BertModel.from_pretrained("bert-base-uncased")

class CaptionBERT(nn.Module):
    def __init__(self, bert_model, hidden_dim=128, num_classes=3, dropout=0.5, freeze_bert=True):
        super().__init__()
        self.bert = bert_model
        if freeze_bert: # prevents BERT weights from being updated
            for param in self.bert.parameters(): # for all trainable tensors in BERT
                param.requires_grad = False      # don't compute gradients for tensor
        
        self.dropout = nn.Dropout(dropout)
        self.fc_hidden = nn.Linear(self.bert.config.hidden_size, hidden_dim) # self.bert.config.hidden_size is 768 for bert-base
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)     # output: last_hidden_state [batch_size, seq_len, hidden_size], pooler_output [batch_size, hidden_size] - a pooled embedding for the [CLS] token
        cls_emb = outputs.last_hidden_state[:,0,:]  # [CLS] token embedding. takes all examples and all features, first token ([CLS]). cls_emb is embedding representing the whole sentence
        x = self.dropout(cls_emb)
        x = self.fc_hidden(x)
        x = self.relu(x)
        x = self.dropout(x)
        out = self.fc_out(x) # raw logits
        return out

# Hyperparameters
hidden_dim = 256 # original 128
learning_rate = 2e-5 # reduced learning rate from 0.001 to 0.00002

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CaptionBERT(bert_model, hidden_dim=hidden_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 


### Train and test loop

In [112]:

# training loop
def train_epoch(loader):
    # Put model in training mode
    model.train()

    # Initialize accumulator for total loss
    total_loss = 0

    # Loops over batches in training set
    for batch in loader:
        # Detect if batch is a dict (BERT-style) or tuple (LSTM/MLP/CNN)
        if isinstance(batch, dict):
            # BERT-style
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            # total_loss += loss.item()
            # correct += (logits.argmax(dim=1) == labels).sum().item()
            # total += labels.size(0)

        else:
            # Tuple-style: (X, y) for LSTM, MLP, CNN
            X, y = batch

            # Move x y to same device
            X, y = X.to(device), y.to(device)

            # Clear old gradients from previous batch
            optimizer.zero_grad()

            # Feeds batch into model, perform forward pass, output logits
            logits = model(X)
            # logits = model(input_ids=X["input_ids"], attention_mask=X["attention_mask"])


            # Compute loss for this batch
            loss = criterion(logits, y)

            # Performs backpropagation: computes gradients of the loss with respect to each model parameter.
            loss.backward()

            # Updates model parameters using those gradients
            optimizer.step()

        # Adds the numeric loss value (converted from a tensor with .item()) to the total loss accumulator.
        total_loss += loss.item()

    # Average loss = total loss / number of batches
    avg_loss = total_loss / len(loader)
    
    return avg_loss




In [113]:
def test_epoch(loader):
    # Set model to evaluation mode. Disables dropout, freezes batchnorm stats
    model.eval()

    # Initialize accumulator for total loss
    total_loss = 0

    with torch.no_grad(): # Disables gradient tracking inside this block.
    # This saves memory and time, since we don’t need gradients for validation.
    # It also ensures the model’s weights won’t accidentally be updated.
        for batch in loader:
            if isinstance(batch, dict):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                y = labels
                # total_loss += criterion(logits, labels).item()
            else:
                X, y = batch
                X, y = X.to(device), y.to(device)
    
                logits = model(X)
                # logits = model(input_ids=X["input_ids"], attention_mask=X["attention_mask"])


            total_loss += criterion(logits, y).item()

    # Compute average loss
    avg_loss = total_loss / len(loader)
    
    return avg_loss

In [114]:
from sklearn.metrics import confusion_matrix, f1_score

def evaluate_metrics(loader, model):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, dict):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids=input_ids, attention_mask=attention_mask)
                y = labels
            else:
                X, y = batch
                X, y = X.to(device), y.to(device)

                logits = model(X)
                # logits = model(input_ids=X["input_ids"], attention_mask=X["attention_mask"])

            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Accuracy
    correct = sum(p == y for p, y in zip(all_preds, all_labels))
    accuracy = correct / len(all_labels)

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Macro F1
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return accuracy, cm, macro_f1


In [115]:
for epoch in range(10):
    train_loss = train_epoch(train_loader)
    test_loss  = test_epoch(test_loader)

    # Metrics
    train_acc, train_cm, train_macro_f1 = evaluate_metrics(train_loader, model)
    test_acc, test_cm, test_macro_f1 = evaluate_metrics(test_loader, model)

    print(f"\nEpoch {epoch+1}")
    print(f"Train   | loss={train_loss:.4f}, acc={train_acc:.3f}, macro-F1={train_macro_f1:.3f}")
    print(f"Test    | loss={test_loss:.4f}, acc={test_acc:.3f}, macro-F1={test_macro_f1:.3f}")

    print("Train Confusion Matrix:")
    print(train_cm)

    print("Test Confusion Matrix:")
    print(test_cm)



Epoch 1
Train   | loss=1.1106, acc=0.334, macro-F1=0.187
Test    | loss=1.1039, acc=0.279, macro-F1=0.150
Train Confusion Matrix:
[[  4 457   1]
 [ 16 474   6]
 [ 10 490  13]]
Test Confusion Matrix:
[[  1 143   3]
 [  6 104   4]
 [  0 115   0]]

Epoch 2
Train   | loss=1.1075, acc=0.377, macro-F1=0.295
Test    | loss=1.1000, acc=0.335, macro-F1=0.277
Train Confusion Matrix:
[[  0 344 118]
 [  2 350 144]
 [  2 307 204]]
Test Confusion Matrix:
[[  2 112  33]
 [  0  84  30]
 [  0  75  40]]

Epoch 3
Train   | loss=1.1038, acc=0.407, macro-F1=0.343
Test    | loss=1.0947, acc=0.356, macro-F1=0.315
Train Confusion Matrix:
[[ 22 247 193]
 [ 14 253 229]
 [  9 181 323]]
Test Confusion Matrix:
[[ 7 82 58]
 [ 2 58 54]
 [ 2 44 69]]

Epoch 4
Train   | loss=1.0976, acc=0.396, macro-F1=0.330
Test    | loss=1.0926, acc=0.335, macro-F1=0.295
Train Confusion Matrix:
[[ 27 189 246]
 [ 21 174 301]
 [ 13 118 382]]
Test Confusion Matrix:
[[ 9 59 79]
 [ 3 40 71]
 [ 3 35 77]]

Epoch 5
Train   | loss=1.0899, ac

Next steps to try:

For LSTM:
* bidirectional
* increase dropout
* pre-trained embeddings

For BERT:
* Unfreeze and fine tune last 2-4 layers